In [ ]:
#!pip install google-colab-selenium

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
#import google_colab_selenium as gs
import requests
from bs4 import BeautifulSoup
import time
import requests
tqdm.pandas()

In [ ]:
!ls data

 batch_api_input		    data.json			 summ_exp_preprocessed.xlsx
 batch_api_output		    data_with_llama_exp.xlsx	 total_data_fin_exp_eval.xlsx
 cf_data_preprocessed.xlsx	    data_with_mistral_exp.xlsx	 total_data_fin_exp.xlsx
'Copy of total_data_fin_exp.xlsx'   gpt4_gen_response.xlsx	 total_data.xlsx
 counter_factual.csv		    prom_eval3_data.xlsx
 Cp_total_data.xlsx		    summarization.tsv


# exploring the conuterfactual data of the paper 'ask more to know more'

In [ ]:
cf_data=pd.read_csv('data/counter_factual.csv').drop(columns='Unnamed: 0')
cf_data=cf_data[['claim','evidence','explanation','counter_factual_Affirmative','counter_factual_Negative','counter_factual_Mixed']]
cf_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   claim                        500 non-null    object
 1   evidence                     500 non-null    object
 2   explanation                  500 non-null    object
 3   counter_factual_Affirmative  500 non-null    object
 4   counter_factual_Negative     500 non-null    object
 5   counter_factual_Mixed        500 non-null    object
dtypes: object(6)
memory usage: 23.6+ KB


In [ ]:
cf_data.head()

,claim,evidence,explanation,counter_factual_Affirmative,counter_factual_Negative,counter_factual_Mixed
0,Adrienne Bailon is an accountant.,Adrienne Eliza Houghton -LRB- née Bailon ; bor...,Adrienne Bailon is an american singer-songwrit...,If we were to say 'Adrienne Bailon is an Ameri...,If we were to say 'Adrienne Bailon IS NOT an a...,If we were to say 'Adrienne Bailon is an accou...
1,"Stranger Things is set in Bloomington, Indiana.","Set in the fictional town of Hawkins , Indiana...","Stranger Things is set in Hawkins, Indiana.",If we were to say 'Stranger Things is set in H...,If we were to say 'Stranger Things IS NOT set ...,If we were to say 'Stranger Things is set in B...
2,Puerto Rico is not an unincorporated territory...,Puerto Rico -LRB- Spanish for `` Rich Port '' ...,It is an unincorporated territory of the unite...,If we were to say 'It is an unincorporated ter...,If we were to say 'Puerto Rico is an unincorpo...,If we were to say 'Puerto Rico is not an uninc...
3,Peggy Sue Got Married is a Egyptian film relea...,Peggy Sue Got Married is a 1986 American comed...,Peggy sue got married is a 1986 american comed...,If we were to say 'Peggy sue got married is a ...,If we were to say 'Peggy Sue Got Married IS NO...,If we were to say 'Peggy Sue Got Married is a ...
4,Andy Roddick lost 5 Master Series between 2002...,Roddick was ranked in the top 10 for nine cons...,Roddick lost the Master Series between 2002 an...,If we were to say 'Roddick lost the Master Ser...,If we were to say 'Andy Roddick DID NOT lose 5...,If we were to say 'Andy Roddick lost 5 Master ...


# Exploring the summarization explanations in fact-checking data of the paper "Explainable Automated Fact-Checking for Public Health Claims"

In [ ]:
summ_exp=pd.read_csv('data/summarization.tsv', sep='\t').dropna()[['claim','explanation', 'main_text','sources', 'label']]
summ_exp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7875 entries, 0 to 9831
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        7875 non-null   object
 1   explanation  7875 non-null   object
 2   main_text    7875 non-null   object
 3   sources      7875 non-null   object
 4   label        7875 non-null   object
dtypes: object(5)
memory usage: 369.1+ KB


In [ ]:
summ_exp.head()

,claim,explanation,main_text,sources,label
0,"""The money the Clinton Foundation took from fr...","""Gingrich said the Clinton Foundation """"took m...","""Hillary Clinton is in the political crosshair...",https://www.wsj.com/articles/clinton-foundatio...,false
1,Annual Mammograms May Have More False-Positives,This article reports on the results of a study...,While the financial costs of screening mammogr...,,mixture
2,SBRT Offers Prostate Cancer Patients High Canc...,This news release describes five-year outcomes...,The news release quotes lead researcher Robert...,https://www.healthnewsreview.org/wp-content/up...,mixture
3,"Study: Vaccine for Breast, Ovarian Cancer Has ...","While the story does many things well, the ove...","The story does discuss costs, but the framing ...",http://clinicaltrials.gov/ct2/results?term=can...,true
4,Some appendicitis cases may not require ’emerg...,We really don’t understand why only a handful ...,"""Although the story didn’t cite the cost of ap...",,true


In [ ]:
summ_exp.label.value_counts()

,count
label,
true,3191
false,2974
mixture,1424
unproven,286


spot the evidence that has resources:

In [ ]:
'http' in str(summ_exp[summ_exp.claim=='Some appendicitis cases may not require ’emergency’ surgery'].sources)

False

#datasets integration

In [ ]:
summ_exp.columns

Index(['claim', 'explanation', 'main_text', 'sources', 'label'], dtype='object')

In [ ]:
cf_data.columns

Index(['claim', 'evidence', 'explanation', 'counter_factual_Affirmative',
       'counter_factual_Negative', 'counter_factual_Mixed'],
      dtype='object')

### unifying the columns between the datasets:

#### first the cf data:

In [ ]:
cf_data['sources']=[pd.NA]*len(cf_data)
cf_data['label']=['false' for i in range(len(cf_data))]

In [ ]:
cf_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   claim                        500 non-null    object
 1   evidence                     500 non-null    object
 2   explanation                  500 non-null    object
 3   counter_factual_Affirmative  500 non-null    object
 4   counter_factual_Negative     500 non-null    object
 5   counter_factual_Mixed        500 non-null    object
 6   sources                      0 non-null      object
 7   label                        500 non-null    object
dtypes: object(8)
memory usage: 31.4+ KB


Adding refrences and citations:

In [ ]:
def get_src_info(query:str) -> str:
  headers = {
      'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
      'accept-language': 'en-US,en;q=0.9',
      #'cache-control': 'max-age=0',
      # 'cookie': 'receive-cookie-deprecation=1; HSID=At64aopg2ebrOFaIa; SSID=A3irvlYOxwBomjDr8; APISID=4Q4JNV0R2Mtvi5n8/A0ichuR5OLQIqAaMW; SAPISID=O9a-UVrgnbg5dJw4/A21t7uW9f76yADHO8; __Secure-1PAPISID=O9a-UVrgnbg5dJw4/A21t7uW9f76yADHO8; __Secure-3PAPISID=O9a-UVrgnbg5dJw4/A21t7uW9f76yADHO8; SEARCH_SAMESITE=CgQI-5sB; SID=g.a000qQj8AZjfqSI57HZy9frWo6mxxbJjwl8OpG9kBBTTZFS-JToQLeX-o00H1ElbFduHkfdNEgACgYKAZQSARESFQHGX2MidZPwvK9LwLgHWUe7ToUtIxoVAUF8yKpA6QsorT8ySKUtiICNyltO0076; __Secure-1PSID=g.a000qQj8AZjfqSI57HZy9frWo6mxxbJjwl8OpG9kBBTTZFS-JToQjGRpHrZCEAIQjaeGeRuTlQACgYKAUMSARESFQHGX2MiAALaZ9YIwKOtHebuecaGrBoVAUF8yKoFy2P-7QA6Xta3_I4bf1eh0076; __Secure-3PSID=g.a000qQj8AZjfqSI57HZy9frWo6mxxbJjwl8OpG9kBBTTZFS-JToQYpfiiZm1CL_hhI1oqGx9LgACgYKAagSARESFQHGX2MiPUtDzFGvsIg0DUj1AF46HhoVAUF8yKrCSlIc03Zn4tifuCQ7BOQT0076; AEC=AZ6Zc-VVe0_X4GGL3PceUQtKhe2yePFtL5dZ93bwjv9dkoTa7Z1Fg5MC6X0; NID=519=i8KDSvVYyzALE5J6YveYOJHh5HRGDmhbiyne3prCnxV8meIO4ACoUOMpBzJ2cuOicQpkR1FPLYKKJoKxlg7l7m5DPi1hbi1T3oozkI3TiEOWw8QrUj0iE5FhIV2uNrPQzunc0hu4gpDsYD6jhQoA-oid0V0ke-o9vwjuIqFCtpy5UxgG0qZ9zAdExYGXmijdNXBrSpiSDKoGch57SCvlgN2CFmi473BAUZonLleoqWGetyRpRpABeCrTDknK6Ti9hWB8Kq9sB1FEcitL6Gnj7XduhpO-5QWvuADC8Tk5HeAzF-7mzQjw67iUvcUVc1KJHOMk7ed3ljXCLq6UifxqBcvHQYMWTG-SQ5Lba_A9WYqPvouVNoszZQQi05tdyBH6bCqEIiblPUKChsRj2XmxwgHbXOm3wSFRVfHgLWBDswOZbBIScTyO-lhE3juR_eNtQA6pHUFb9D49DRcUhja-7d9qJAi1wTDWP9tkJdd9BEmLHGt2JHcr5tN7TE5NRn3XgCKyoeDfNJq6kkYpSQ; __Secure-1PSIDTS=sidts-CjEBQT4rX4GtGt5IILTv5lNCBe_Tg1lX0YXPpkUhfy8ov8Wr3AhgmfS4S1hUM8fceGoSEAA; __Secure-3PSIDTS=sidts-CjEBQT4rX4GtGt5IILTv5lNCBe_Tg1lX0YXPpkUhfy8ov8Wr3AhgmfS4S1hUM8fceGoSEAA; SIDCC=AKEyXzWLwFlXuovL77B0eINGUDfiOujAU8u8TWLi4QY3dz1fm3iy5Gf8hpjKS0UNZfdQAyKdhhw; __Secure-1PSIDCC=AKEyXzUS-r6t_MjOIExcczO_8ox8iU_2S4vufGi_Zd1FsVn2iTephVAhTIoh_CB6MM3BoryTETOY; __Secure-3PSIDCC=AKEyXzWF5WUgE8-GIOvm49ezco9fQ9Wlfu7rrBE7B_gyvBLazVIMnX5C6Lds7vQyrm-aV-stZr4',
      'priority': 'u=0, i',
      'referer': 'https://www.google.com/',
      'sec-ch-prefers-color-scheme': 'light',
      'sec-ch-ua': '"Chromium";v="130", "Google Chrome";v="130", "Not?A_Brand";v="99"',
      'sec-ch-ua-arch': '"x86"',
      'sec-ch-ua-bitness': '"64"',
      'sec-ch-ua-form-factors': '"Desktop"',
      'sec-ch-ua-full-version': '"130.0.6723.119"',
      'sec-ch-ua-full-version-list': '"Chromium";v="130.0.6723.119", "Google Chrome";v="130.0.6723.119", "Not?A_Brand";v="99.0.0.0"',
      'sec-ch-ua-mobile': '?0',
      'sec-ch-ua-model': '""',
      'sec-ch-ua-platform': '"Windows"',
      #'sec-ch-ua-platform-version': '"15.0.0"',
      'sec-ch-ua-wow64': '?0',
      'sec-fetch-dest': 'document',
      'sec-fetch-mode': 'navigate',
      'sec-fetch-site': 'same-origin',
      'sec-fetch-user': '?1',
      'upgrade-insecure-requests': '1',
      'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36',
      'x-browser-channel': 'stable',
      #'x-browser-copyright': 'Copyright 2024 Google LLC. All rights reserved.',
      #'x-browser-validation': 'QTcFh+5ZedCCidQkKanqiUwPHQo=',
      #'x-browser-year': '2024',
      '#x-client-data': 'CI22yQEIorbJAQipncoBCNHfygEIlqHLAQiFoM0BCLrIzQEIsZ7OAQiMxs4BCMbMzgEIxs/OAQj0z84BCK3QzgEYwcvMARidsc4BGJPQzgE=',
  }

  params = {
      'q': query,
      'sca_esv': '74c6de03b5b81e4a',
      'sxsrf': 'ADLYWIIV1jPhY0Uu5T5rncPWulX43fNcOQ:1733071510010',
      'ei': 'lpJMZ6IZ76Dk2g-l54KQAg',
      'ved': '0ahUKEwiivOW9goeKAxVvEFkFHaWzACIQ4dUDCA8',
      'uact': '5',
      'oq': query,
      'gs_lp': 'Egxnd3Mtd2l6LXNlcnAi9wFQZWdneSBTdWUgR290IE1hcnJpZWQgaXMgYSAxOTg2IEFtZXJpY2FuIGNvbWVkeS1kcmFtYSBmaWxtIGRpcmVjdGVkIGJ5IEZyYW5jaXMgRm9yZCBDb3Bwb2xhIHN0YXJyaW5nIEthdGhsZWVuIFR1cm5lciBhcyBhIHdvbWFuIG9uIHRoZSB2ZXJnZSBvZiBhIGRpdm9yY2UgLCB3aG8gZmluZHMgaGVyc2VsZiB0cmFuc3BvcnRlZCBiYWNrIHRvIHRoZSBkYXlzIG9mIGhlciBzZW5pb3IgeWVhciBpbiBoaWdoIHNjaG9vbCBpbiAxOTYwIC4JSABQAFgAcAB4AZABAJgBAKABAKoBALgBA8gBAPgBAvgBAZgCAKACAJgDAJIHAKAHAA',
      'sclient': 'gws-wiz-serp',
  }

  response = requests.get('https://www.google.com/search', params=params, headers=headers)
  time.sleep(1)
  try:
    return str(BeautifulSoup(response.text).find_all("div",{"class": "N54PNb BToiNc"})[0].find_all("a")[0]).split('href="')[1].split('" jsname')[0]
  except:
    return "connection error with code " + str(response.status_code)

In [ ]:
# def get_src_info(src_names:list) -> list:
#   '''This function gets a list of paper names and produces a list of the paper authors, links and correct names. the
#   target is generate the correct resources without errors.
#   input: list of paper names to get their correct citations
#   output: corresponding list of paper names after verification along with some of their authors and their most recent links'''
#   driver = gs.Chrome()
#   srcs_correct_names,  srcs_correct_links, srcs_correct_authors = [], [], []
#   for src in src_names: #get the src name
#     driver.get("https://scholar.google.ca/scholar?hl=en&as_sdt=0%2C5&as_vis=1&q={}".format(src))
#     response = driver.page_source
#     try:
#       src_name = BeautifulSoup(response).find_all("div", {"class": "gs_ri"})[0].find_all("a")[0].text
#       src_link = BeautifulSoup(response).find_all("div", {"class": "gs_ri"})[0].find_all("a")[0]['href']
#     except:
#       return [response]
#     src_authors = []
#     for element in BeautifulSoup(response).find_all("div", {"class": "gs_ri"})[0].find_all("a"):
#       if "citations" in str(element) and element.text not in src_authors:
#         src_authors.append(element.text)
#     srcs_correct_names.append(src_name)
#     srcs_correct_links.append(src_link)
#     srcs_correct_authors.append(src_authors)
#     time.sleep(1)
#   driver.quit()
#   #return srcs_correct_names, srcs_correct_links, srcs_correct_authors
#   return srcs_correct_links

In [ ]:
get_src_info('The rise of fact-checking sites in')

'https://reutersinstitute.politics.ox.ac.uk/our-research/rise-fact-checking-sites-europe'

In [ ]:
get_src_info(cf_data.evidence[148])

'https://scholar.harvard.edu/tcheng3/news/news-006'

#### Fetching the sources, and combining different CF explanations and adding metadata:

In [ ]:
metadata=[]
exp=[]
srcs=[]
for i in tqdm(range(len(cf_data))):
  if i < 167:
    exp.append(cf_data.explanation[i])
    if i < 84:
      metadata.append('error_correction_only')
      srcs.append(pd.NA)
    else:
      metadata.append('error_correction_only_with_src')
      srcs.append(get_src_info(cf_data.evidence[i]))

  elif i >=167 and i < 334:
    exp.append('error is detected in the part: ' + cf_data.counter_factual_Affirmative[i].split('instead of ')[1].split("',")[0].strip("'"))
    if i < 251:
      metadata.append('error_detection_only')
      srcs.append(pd.NA)
    else:
      metadata.append('error_detection_only_with_src')
      srcs.append(get_src_info(cf_data.evidence[i]))

  elif i >= 334 and i < 389:
    exp.append(cf_data.counter_factual_Affirmative[i])
    if i <362:
      metadata.append('error_detection_and_correction_only')
      srcs.append(pd.NA)
    else:
      metadata.append('error_detection_and_correction_with_src')
      srcs.append(get_src_info(cf_data.evidence[i]))

  elif i >= 389 and i < 444:
    exp.append(cf_data.counter_factual_Negative[i])
    if i < 417:
      metadata.append('error_detection_and_correction_only')
      srcs.append(pd.NA)
    else:
      metadata.append('error_detection_and_correction_with_src')
      srcs.append(get_src_info(cf_data.evidence[i]))

  else:
    exp.append(cf_data.counter_factual_Mixed[i])
    if i < 472:
      metadata.append('error_detection_and_correction_only')
      srcs.append(pd.NA)
    else:
      metadata.append('error_detection_and_correction_with_src')
      srcs.append(get_src_info(cf_data.evidence[i]))


100%|██████████| 500/500 [06:58<00:00,  1.20it/s]


In [ ]:
cf_data['exp']=exp
cf_data['metadata']=metadata
cf_data['sources']=srcs

In [ ]:
cf_data.drop(columns=['explanation', 'counter_factual_Affirmative', 'counter_factual_Negative', 'counter_factual_Mixed'], inplace=True)
cf_data.rename(columns={'exp':'explanation'}, inplace=True)
cf_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        500 non-null    object
 1   evidence     500 non-null    object
 2   sources      248 non-null    object
 3   label        500 non-null    object
 4   explanation  500 non-null    object
 5   metadata     500 non-null    object
dtypes: object(6)
memory usage: 23.6+ KB


In [ ]:
cf_data.to_excel('data/cf_data_preprocessed.xlsx', index=False)

#### second the summ data

In [ ]:
summ_exp.rename(columns={'main_text':'evidence'}, inplace=True)
summ_exp=summ_exp[summ_exp.label.str.contains('true|false')]
summ_exp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6165 entries, 0 to 9831
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        6165 non-null   object
 1   explanation  6165 non-null   object
 2   evidence     6165 non-null   object
 3   sources      6165 non-null   object
 4   label        6165 non-null   object
dtypes: object(5)
memory usage: 289.0+ KB


In [ ]:
summ_exp.label.value_counts()

,count
label,
true,3191
false,2974


In [ ]:
true_resourced=summ_exp[(summ_exp.label=='true') & (summ_exp.sources.str.contains('http'))].head(166)
false_resourced=summ_exp[(summ_exp.label=='false')& (summ_exp.sources.str.contains('http'))].head(166)
true_nonresourced=summ_exp[(summ_exp.label=='true') & (~summ_exp.sources.str.contains('http'))].head(166)
false_nonresourced=summ_exp[(summ_exp.label=='false') & (~summ_exp.sources.str.contains('http'))].head(166)

In [ ]:
true_resourced['metadata']=['summarization_true_label']*len(true_resourced)
true_nonresourced['metadata']=['summarization_false_label']*len(true_nonresourced)
false_resourced['metadata']=['summarization_true_label']*len(false_resourced)
false_nonresourced['metadata']=['summarization_false_label']*len(false_nonresourced)

In [ ]:
false_nonresourced['sources']=[pd.NA]*len(false_nonresourced)
true_nonresourced['sources']=[pd.NA]*len(true_nonresourced)

In [ ]:
summ_exp=pd.concat([true_resourced, true_nonresourced, false_resourced, false_nonresourced], ignore_index=True)
summ_exp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 664 entries, 0 to 663
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        664 non-null    object
 1   explanation  664 non-null    object
 2   evidence     664 non-null    object
 3   sources      332 non-null    object
 4   label        664 non-null    object
 5   metadata     664 non-null    object
dtypes: object(6)
memory usage: 31.2+ KB


In [ ]:
summ_exp.to_excel('data/summ_exp_preprocessed.xlsx', index=False)

#### integrating both datasets in one file:

In [ ]:
cf_data=pd.read_excel('data/cf_data_preprocessed.xlsx')
summ_exp=pd.read_excel('data/summ_exp_preprocessed.xlsx')

In [ ]:
total_data=pd.concat([cf_data, summ_exp], ignore_index=True)
total_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        1164 non-null   object
 1   evidence     1164 non-null   object
 2   sources      580 non-null    object
 3   label        1164 non-null   bool  
 4   explanation  1164 non-null   object
 5   metadata     1164 non-null   object
dtypes: bool(1), object(5)
memory usage: 46.7+ KB


In [ ]:
total_data.to_excel('data/total_data.xlsx', index=False)

# dealing with data after adding mistral explanations

In [ ]:
data=pd.read_excel('data/data_with_mistral_exp.xlsx')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        1164 non-null   object
 1   evidence     1164 non-null   object
 2   sources      580 non-null    object
 3   label        1164 non-null   bool  
 4   explanation  1164 non-null   object
 5   metadata     1164 non-null   object
 6   mistral_exp  1164 non-null   object
dtypes: bool(1), object(6)
memory usage: 55.8+ KB


In [ ]:
def extract_exp(exp):
  return exp.split('[/INST]')[-1].replace('</s>', '').strip()

In [ ]:
data['mistral_exp']=data.mistral_exp.progress_apply(extract_exp)

100%|██████████| 1164/1164 [00:00<00:00, 91520.66it/s]


In [ ]:
data.to_excel('data/total_data.xlsx', index=False)

# integrating the llama explanations

In [ ]:
import pandas as pd

In [ ]:
data=pd.read_excel('data/data_with_llama_exp.xlsx')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        1164 non-null   object
 1   evidence     1164 non-null   object
 2   sources      580 non-null    object
 3   label        1164 non-null   bool  
 4   explanation  1164 non-null   object
 5   metadata     1164 non-null   object
 6   llama_exp    1164 non-null   object
dtypes: bool(1), object(6)
memory usage: 55.8+ KB


In [ ]:
data.head()

,claim,evidence,sources,label,explanation,metadata,llama_exp
0,Adrienne Bailon is an accountant.,Adrienne Eliza Houghton -LRB- née Bailon ; bor...,NaN,False,Adrienne Bailon is an american singer-songwrit...,error_correction_only,"Based on the evidence, the claim ""Adrienne Bai..."
1,"Stranger Things is set in Bloomington, Indiana.","Set in the fictional town of Hawkins , Indiana...",NaN,False,"Stranger Things is set in Hawkins, Indiana.",error_correction_only,"Actionable Explanation:\n\nThe claim ""Stranger..."
2,Puerto Rico is not an unincorporated territory...,Puerto Rico -LRB- Spanish for `` Rich Port '' ...,NaN,False,It is an unincorporated territory of the unite...,error_correction_only,Actionable Explanation:\n\nThe claim that Puer...
3,Peggy Sue Got Married is a Egyptian film relea...,Peggy Sue Got Married is a 1986 American comed...,NaN,False,Peggy sue got married is a 1986 american comed...,error_correction_only,"Based on the provided evidence, the claim ""Peg..."
4,Andy Roddick lost 5 Master Series between 2002...,Roddick was ranked in the top 10 for nine cons...,NaN,False,Roddick lost the Master Series between 2002 an...,error_correction_only,"Based on the evidence, the claim that Andy Rod..."


In [ ]:
total_data=pd.read_excel('data/total_data.xlsx')
total_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        1164 non-null   object
 1   evidence     1164 non-null   object
 2   sources      580 non-null    object
 3   label        1164 non-null   bool  
 4   explanation  1164 non-null   object
 5   metadata     1164 non-null   object
 6   mistral_exp  1164 non-null   object
dtypes: bool(1), object(6)
memory usage: 55.8+ KB


In [ ]:
new_data=pd.merge(total_data, data, on=['claim','evidence','sources','label','explanation','metadata'], how='left')
new_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        1164 non-null   object
 1   evidence     1164 non-null   object
 2   sources      580 non-null    object
 3   label        1164 non-null   bool  
 4   explanation  1164 non-null   object
 5   metadata     1164 non-null   object
 6   mistral_exp  1164 non-null   object
 7   llama_exp    1164 non-null   object
dtypes: bool(1), object(7)
memory usage: 64.9+ KB


In [ ]:
new_data.to_excel('data/total_data.xlsx', index=False)

#converting the data into json objects

In [ ]:
import json

In [ ]:
df1=pd.read_excel('data/total_data.xlsx')
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        1164 non-null   object
 1   evidence     1164 non-null   object
 2   sources      580 non-null    object
 3   label        1164 non-null   bool  
 4   explanation  1164 non-null   object
 5   metadata     1164 non-null   object
 6   mistral_exp  1164 non-null   object
 7   llama_exp    1164 non-null   object
dtypes: bool(1), object(7)
memory usage: 64.9+ KB


In [ ]:
d = df1.to_dict(orient='records')

In [ ]:
with open('data/data.json', 'w') as f:
    json.dump(d, f)

# calculating when we use GPT for input tokens

In [ ]:
with open('data/data.json') as f:
    d = json.load(f)


In [ ]:
(len(str(d).split())/750)*0.005

6.3894866666666665

In [ ]:
len(str(d).split())

958423

# adding gpt evaluations

In [ ]:
with open('gpt_4_response.txt') as f:
    d = list(f)

In [ ]:
len(d)

1164

In [ ]:
d[:3]

['index: 0;;;;[3,4,5,5] \n',
 'index: 1;;;;[3, 5, 5, 5]\n',
 'index: 2;;;;[3, 4, 5, 4]\n']

In [ ]:
ind=[int(item.split(';;;;')[0].replace('index: ','')) for item in d]

In [ ]:
len(ind)

1164

In [ ]:
for i in range(1164):
  if i not in ind:
    print(i)

In [ ]:
ind[-5:]

[1159, 1160, 1161, 1162, 1163]

In [ ]:
val=[item.split(';;;;')[1].replace('\n','').replace('[','').replace(']','').split(',') for item in d]
len(val)

1164

In [ ]:
gpt_expl1 = [int(item[0]) for item in val]
gpt_expl2 = [int(item[1]) for item in val]
gpt_expl3 = [int(item[2]) for item in val]
gpt_expl4 = [int(item[3]) for item in val]

In [ ]:
len(gpt_expl3)

1164

In [ ]:
gpt_expl4[:5]

[5, 5, 4, 3, 5]

In [ ]:
gpt_expl2[:5]

[4, 5, 4, 4, 5]

In [ ]:
val[:5]

[['3', '4', '5', '5 '],
 ['3', ' 5', ' 5', ' 5'],
 ['3', ' 4', ' 5', ' 4'],
 ['1', ' 4', ' 5', ' 3'],
 ['1', ' 5', ' 5', ' 5']]

In [ ]:
df = pd.read_excel('data/total_data_fin_exp.xlsx')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        1164 non-null   object
 1   evidence     1164 non-null   object
 2   sources      580 non-null    object
 3   label        1164 non-null   bool  
 4   explanation  1164 non-null   object
 5   metadata     1164 non-null   object
 6   mistral_exp  1164 non-null   object
 7   llama_exp    1164 non-null   object
 8   gpt-4_exp    1164 non-null   object
dtypes: bool(1), object(8)
memory usage: 74.0+ KB


In [ ]:
df['geval_exp_eval']=gpt_expl1
df['geval_mist_exp_eval']=gpt_expl2
df['geval_llama_exp_eval']=gpt_expl3
df['geval_gpt_exp_eval']=gpt_expl4

In [ ]:
df.to_excel('data/total_data_fin_exp.xlsx', index=False)

#adding prometheus evaluations

In [ ]:
import pandas as pd

In [ ]:
%cd data

/content/drive/MyDrive/second_paper_insha_allah/data


In [ ]:
!ls

 batch_api_input		    Cp_total_data.xlsx		 prom_eval3_data.xlsx
 batch_api_output		    data.json			 summarization.tsv
 cf_data_preprocessed.xlsx	    data_with_llama_exp.xlsx	 summ_exp_preprocessed.xlsx
'Copy of total_data_fin_exp.xlsx'   data_with_mistral_exp.xlsx	 total_data_fin_exp.xlsx
 counter_factual.csv		    gpt4_gen_response.xlsx	 total_data.xlsx


In [ ]:
df=pd.read_excel('total_data_fin_exp.xlsx')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   claim                          1164 non-null   object
 1   evidence                       1164 non-null   object
 2   sources                        580 non-null    object
 3   label                          1164 non-null   bool  
 4   metadata                       1164 non-null   object
 5   explanation                    1164 non-null   object
 6   mistral_exp                    1164 non-null   object
 7   llama_exp                      1164 non-null   object
 8   gpt-4_exp                      1164 non-null   object
 9   geval_exp_eval                 1164 non-null   int64 
 10  geval_mist_exp_eval            1164 non-null   int64 
 11  geval_llama_exp_eval           1164 non-null   int64 
 12  geval_gpt_exp_eval             1164 non-null   int64 
 13  pro

In [ ]:
d=pd.read_excel('prom_eval3_data.xlsx')
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   claim           1164 non-null   object
 1   evidence        1164 non-null   object
 2   sources         580 non-null    object
 3   label           1164 non-null   bool  
 4   explanation     1164 non-null   object
 5   metadata        1164 non-null   object
 6   mistral_exp     1164 non-null   object
 7   llama_exp       1164 non-null   object
 8   prom_ans        1164 non-null   object
 9   prom_ans_mist   1164 non-null   object
 10  prom_ans_llama  1164 non-null   object
dtypes: bool(1), object(10)
memory usage: 92.2+ KB


In [ ]:
prom_eval_exp_justified=d.prom_ans.tolist()
prom_eval_mist_exp_justified=d.prom_ans_mist.tolist()
prom_eval_llama_exp_justified=d.prom_ans_llama.tolist()

In [ ]:
df['prom_eval_exp_justified']=prom_eval_exp_justified
df['prom_eval_mist_exp_justified']=prom_eval_mist_exp_justified
df['prom_eval_llama_exp_justified']=prom_eval_llama_exp_justified

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   claim                          1164 non-null   object
 1   evidence                       1164 non-null   object
 2   sources                        580 non-null    object
 3   label                          1164 non-null   bool  
 4   metadata                       1164 non-null   object
 5   explanation                    1164 non-null   object
 6   mistral_exp                    1164 non-null   object
 7   llama_exp                      1164 non-null   object
 8   gpt-4_exp                      1164 non-null   object
 9   geval_exp_eval                 1164 non-null   int64 
 10  geval_mist_exp_eval            1164 non-null   int64 
 11  geval_llama_exp_eval           1164 non-null   int64 
 12  geval_gpt_exp_eval             1164 non-null   int64 
 13  pro

In [ ]:
df.to_excel('total_data_fin_exp.xlsx', index=False)

# formatting the mistral responses and evaluations

In [ ]:
df=pd.read_excel('total_data_fin_exp.xlsx')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   claim                          1164 non-null   object
 1   evidence                       1164 non-null   object
 2   sources                        580 non-null    object
 3   label                          1164 non-null   bool  
 4   metadata                       1164 non-null   object
 5   explanation                    1164 non-null   object
 6   mistral_exp                    1164 non-null   object
 7   llama_exp                      1164 non-null   object
 8   gpt-4_exp                      1164 non-null   object
 9   geval_exp_eval                 1164 non-null   int64 
 10  geval_mist_exp_eval            1164 non-null   int64 
 11  geval_llama_exp_eval           1164 non-null   int64 
 12  geval_gpt_exp_eval             1164 non-null   int64 
 13  pro

In [ ]:
d=pd.read_excel('prom_eval3_data.xlsx')
d.info()

In [ ]:
def get_result_from_feedback(feedback):
  return int(feedback.split('RESULT]')[-1].replace('</s>','').replace(')',''))

In [ ]:
get_result_from_feedback(df.prom_eval_exp_justified[486])

3

In [ ]:
df['prom_eval_exp']=df.prom_eval_exp_justified.progress_apply(get_result_from_feedback)
df['prom_eval_mist_exp']=df.prom_eval_mist_exp_justified.progress_apply(get_result_from_feedback)
df['prom_eval_llama_exp']=df.prom_eval_llama_exp_justified.progress_apply(get_result_from_feedback)

100%|██████████| 1164/1164 [00:00<00:00, 253342.84it/s]


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   claim                          1164 non-null   object
 1   evidence                       1164 non-null   object
 2   sources                        580 non-null    object
 3   label                          1164 non-null   bool  
 4   metadata                       1164 non-null   object
 5   explanation                    1164 non-null   object
 6   mistral_exp                    1164 non-null   object
 7   llama_exp                      1164 non-null   object
 8   gpt-4_exp                      1164 non-null   object
 9   geval_exp_eval                 1164 non-null   int64 
 10  geval_mist_exp_eval            1164 non-null   int64 
 11  geval_llama_exp_eval           1164 non-null   int64 
 12  geval_gpt_exp_eval             1164 non-null   int64 
 13  pro

#adding gpt evaluations

In [ ]:
df=pd.read_excel('data/total_data_fin_exp_eval.xlsx')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   claim                          1164 non-null   object
 1   evidence                       1164 non-null   object
 2   sources                        580 non-null    object
 3   label                          1164 non-null   bool  
 4   metadata                       1164 non-null   object
 5   explanation                    1164 non-null   object
 6   mistral_exp                    1164 non-null   object
 7   llama_exp                      1164 non-null   object
 8   gpt-4_exp                      1164 non-null   object
 9   geval_exp_eval                 1164 non-null   int64 
 10  geval_mist_exp_eval            1164 non-null   int64 
 11  geval_llama_exp_eval           1164 non-null   int64 
 12  geval_gpt_exp_eval             1164 non-null   int64 
 13  pro

In [ ]:
d=pd.read_excel("data/prom_eval_gpt_data.xlsx")
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   claim        1164 non-null   object
 1   evidence     1164 non-null   object
 2   sources      580 non-null    object
 3   label        1164 non-null   bool  
 4   explanation  1164 non-null   object
 5   metadata     1164 non-null   object
 6   mistral_exp  1164 non-null   object
 7   llama_exp    1164 non-null   object
 8   gpt-4_exp    1164 non-null   object
 9   prom_ans     1164 non-null   object
dtypes: bool(1), object(9)
memory usage: 83.1+ KB


In [ ]:
prom_ans=d.prom_ans.tolist()
df['prom_gpt_eval_exp_justified']=prom_ans

In [ ]:
def get_result_from_feedback(feedback):
  try:
    return int(feedback.split('RESULT]')[-1].replace('</s>','').replace(')',''))
  except:
    return int(feedback.split('Score: ')[-1].replace('</s>','').replace(')',''))

In [ ]:
df['prom_gpt_eval_exp']=df.prom_gpt_eval_exp_justified.progress_apply(get_result_from_feedback)

100%|██████████| 1164/1164 [00:00<00:00, 179914.87it/s]


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   claim                          1164 non-null   object
 1   evidence                       1164 non-null   object
 2   sources                        580 non-null    object
 3   label                          1164 non-null   bool  
 4   metadata                       1164 non-null   object
 5   explanation                    1164 non-null   object
 6   mistral_exp                    1164 non-null   object
 7   llama_exp                      1164 non-null   object
 8   gpt-4_exp                      1164 non-null   object
 9   geval_exp_eval                 1164 non-null   int64 
 10  geval_mist_exp_eval            1164 non-null   int64 
 11  geval_llama_exp_eval           1164 non-null   int64 
 12  geval_gpt_exp_eval             1164 non-null   int64 
 13  pro

In [ ]:
df.columns

Index(['claim', 'evidence', 'sources', 'label', 'metadata', 'explanation',
       'mistral_exp', 'llama_exp', 'gpt-4_exp', 'geval_exp_eval',
       'geval_mist_exp_eval', 'geval_llama_exp_eval', 'geval_gpt_exp_eval',
       'prom_eval_exp_justified', 'prom_eval_mist_exp_justified',
       'prom_eval_llama_exp_justified', 'prom_eval_exp', 'prom_eval_mist_exp',
       'prom_eval_llama_exp', 'prom_gpt_eval_exp_justified',
       'prom_gpt_eval_exp'],
      dtype='object')

In [ ]:
df=df[['claim', 'evidence', 'sources', 'label', 'metadata', 'explanation',
       'mistral_exp', 'llama_exp', 'gpt-4_exp', 'geval_exp_eval',
       'geval_mist_exp_eval', 'geval_llama_exp_eval', 'geval_gpt_exp_eval',
       'prom_eval_exp_justified', 'prom_eval_mist_exp_justified',
       'prom_eval_llama_exp_justified', 'prom_gpt_eval_exp_justified' , 'prom_eval_exp', 'prom_eval_mist_exp',
       'prom_eval_llama_exp',
       'prom_gpt_eval_exp']]

In [ ]:
df.to_excel('data/total_data_fin_exp.xlsx', index=False)

In [ ]:
pub_df=df[['claim', 'evidence', 'sources', 'label', 'metadata', 'explanation',
       'mistral_exp', 'llama_exp', 'gpt-4_exp', 'geval_exp_eval',
       'geval_mist_exp_eval', 'geval_llama_exp_eval', 'geval_gpt_exp_eval', 'prom_eval_exp', 'prom_eval_mist_exp',
       'prom_eval_llama_exp',
       'prom_gpt_eval_exp']]

In [ ]:
pub_df.to_excel('data/total_data.xlsx', index=False)

# formatting mistral

In [ ]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

In [ ]:
%cd /content/drive/MyDrive/second_paper_insha_allah

/content/drive/MyDrive/second_paper_insha_allah


In [ ]:
data = pd.read_excel('data/sample_data_fin_evals.xlsx')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   claim                   203 non-null    object
 1   evidence                203 non-null    object
 2   sources                 87 non-null     object
 3   label                   203 non-null    bool  
 4   metadata                203 non-null    object
 5   explanation             203 non-null    object
 6   Human_annotation_exp    203 non-null    object
 7   mistral_exp             203 non-null    object
 8   Human_annotation_exp.1  203 non-null    object
 9   llama_exp               203 non-null    object
 10  Human_annotation_exp.2  203 non-null    object
 11  gpt-4_exp               203 non-null    object
 12  Human_annotation_exp.3  203 non-null    object
 13  geval_eval              203 non-null    object
 14  prom_eval_exp           203 non-null    object
 15  prom_e

In [ ]:
def extract_score(feedback):
  try:
    return int(feedback.split('RESULT]')[-1].replace('</s>', '').strip())
  except:
    return int(feedback.split('Score: ')[-1].replace('</s>', '').strip())

In [ ]:
a=data.prom_eval_exp.progress_apply(extract_score).tolist()
b=data.prom_eval_mist_exp.progress_apply(extract_score).tolist()
c=data.prom_eval_llama_exp.progress_apply(extract_score).tolist()
d=data.prom_eval_gpt_exp.progress_apply(extract_score).tolist()

100%|██████████| 203/203 [00:00<00:00, 182127.00it/s]


In [ ]:
answers=[]
for i in range(len(a)):
  answers.append([a[i],b[i],c[i],d[i]])

In [ ]:
answers[:5]

[[1, 3, 4, 5], [3, 2, 4, 4], [2, 4, 3, 4], [1, 3, 3, 3], [2, 4, 3, 3]]

In [ ]:
data['prom_eval']=answers

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   claim                   203 non-null    object
 1   evidence                203 non-null    object
 2   sources                 87 non-null     object
 3   label                   203 non-null    bool  
 4   metadata                203 non-null    object
 5   explanation             203 non-null    object
 6   Human_annotation_exp    203 non-null    object
 7   mistral_exp             203 non-null    object
 8   Human_annotation_exp.1  203 non-null    object
 9   llama_exp               203 non-null    object
 10  Human_annotation_exp.2  203 non-null    object
 11  gpt-4_exp               203 non-null    object
 12  Human_annotation_exp.3  203 non-null    object
 13  geval_eval              203 non-null    object
 14  prom_eval_exp           203 non-null    object
 15  prom_e

In [ ]:
data.to_excel('data/sample_data_fin_evals.xlsx',index=False)

##getting prometheuse results with links

In [ ]:
df1=pd.read_excel("data/prom_eval_sample_data_with_links.xlsx")
prom_eval_exp_with_links=df1.prom_eval_exp_with_links.to_list()

In [ ]:
df2=pd.read_excel("data/prom_eval_sample_data_with_links2.xlsx")
prom_eval_llama_exp_with_links=df2.prom_eval_llama_exp_with_links
prom_eval_gpt_exp_with_links=df2.prom_eval_gpt_exp_with_links

In [ ]:
df3=pd.read_excel("data/prom_eval_sample_data_with_links3.xlsx")
prom_eval_mistral_exp_with_links=df3.prom_eval_mistral_exp_with_links

In [ ]:
data=pd.read_excel("data/sample_data_fin_evals_exp3.xlsx")
data.info()

In [ ]:
def get_result_from_feedback(feedback):
  return int(feedback.split('RESULT]')[-1].replace('</s>','').replace(')',''))

In [ ]:
data['prom_feedback_exp_with_links']=df1.prom_eval_exp_with_links
data['prom_feedback_exp_mistral_exp_with_links']=df3.prom_eval_mistral_exp_with_links
data['prom_feedback_exp_llama_exp_with_links']=df2.prom_eval_llama_exp_with_links
data['prom_feedback_exp_gpt_exp_with_links']=df2.prom_eval_gpt_exp_with_links

In [ ]:
data.to_excel("data/sample_data_fin_evals_exp3.xlsx", index=False)

In [ ]:
prom_score_exp_with_links=data.prom_feedback_exp_with_links.progress_apply(get_result_from_feedback).to_list()
prom_score_mistral_exp_with_links=data.prom_feedback_exp_mistral_exp_with_links.progress_apply(get_result_from_feedback).to_list()
prom_score_llama_exp_with_links=data.prom_feedback_exp_llama_exp_with_links.progress_apply(get_result_from_feedback).to_list()
prom_score_gpt_exp_with_links=data.prom_feedback_exp_gpt_exp_with_links.progress_apply(get_result_from_feedback).to_list()

100%|██████████| 203/203 [00:00<00:00, 167937.62it/s]


In [ ]:
prom_results_with_links=[]
for i in range(len(data)):
  prom_results_with_links.append([prom_score_exp_with_links[i], prom_score_mistral_exp_with_links[i], prom_score_llama_exp_with_links[i], prom_score_gpt_exp_with_links[i]])

In [ ]:
data["prom_eval_with_links"]=prom_results_with_links
data.to_excel("data/sample_data_fin_evals_exp3.xlsx", index=False)